In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
catalog = "ete"
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"
data_source = "orders"

base_path = f"s3a://S3 Path/{data_source}"
landing_path = f"{base_path}/landing"
processed_path = f"{base_path}/processed"

bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
staging_table = f"{catalog}.{bronze_schema}.staging_{data_source}"

In [0]:
df_new_orders = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(f"{landing_path}/*.csv")
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

print(f"New Incremental Rows Ingested: {df_new_orders.count()}")
display(df_new_orders.limit(5))

In [0]:
df_new_orders.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("append") \
    .saveAsTable(bronze_table)

In [0]:
df_new_orders.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(staging_table)

print(f"Appended to {bronze_table} and staged in {staging_table}")

In [0]:
files = dbutils.fs.ls(landing_path)
moved_count = 0
for file_info in files:
    if file_info.name.endswith(".csv"):
        dbutils.fs.mv(file_info.path, f"{processed_path}/{file_info.name}", True)
        moved_count += 1

print(f" {moved_count} files moved to the processed directory to prevent double-loading.")

## Silver Processing...

In [0]:
df_staging = spark.table(f"{catalog}.{bronze_schema}.staging_{data_source}")
df_cleaned = (
    df_staging
    .filter(F.col("order_qty").isNotNull())
    
    .withColumn("customer_id", F.col("customer_id").cast("string"))
    .withColumn("product_id", F.col("product_id").cast("string"))
    .withColumn("order_placement_date", F.col("order_placement_date").cast("string"))
    
   .withColumn(
        "order_placement_date",
        F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
    )
    .withColumn(
        "order_placement_date",
        F.coalesce(
            F.try_to_date(F.col("order_placement_date"), "yyyy-MM-dd"), 
            F.try_to_date(F.col("order_placement_date"), "yyyy/MM/dd"),
            F.try_to_date(F.col("order_placement_date"), "dd-MM-yyyy"),
            F.try_to_date(F.col("order_placement_date"), "dd/MM/yyyy"),
            F.try_to_date(F.col("order_placement_date"), "MMMM dd, yyyy")
        )
    )
    
    .dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])
)

In [0]:
df_products = spark.table(f"{catalog}.{silver_schema}.products")

df_joined = df_cleaned.join(
    df_products, 
    df_cleaned["product_id"] == df_products["product_code"], 
    how="inner").select(
    df_cleaned["order_id"],
    df_cleaned["order_placement_date"],
    df_cleaned["customer_id"],
    df_cleaned["product_id"].alias("product_code"),
    df_cleaned["order_qty"],
    df_cleaned["read_timestamp"],
    df_cleaned["file_name"],
    df_cleaned["file_size"]
)

print("Cleaned Incremental Data:")
display(df_joined.limit(5))

In [0]:
staging_silver_table = f"{catalog}.{silver_schema}.staging_{data_source}"

df_joined.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(staging_silver_table)

print(f" Cleaned incremental data staged to {staging_silver_table}")

In [0]:
silver_table = f"{catalog}.{silver_schema}.{data_source}"
silver_delta = DeltaTable.forName(spark, silver_table)

silver_delta.alias("silver").merge(
    df_joined.alias("bronze"), 
    "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print(f" Incremental data perfectly merged into main table: {silver_table}")

## Gold Processing...

In [0]:
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}" 
staging_silver_table = f"{catalog}.{silver_schema}.staging_{data_source}"

In [0]:
df_gold = spark.sql(f"""
    SELECT 
        order_id, 
        order_placement_date as date, 
        customer_id as customer_code, 
        product_code, 
        order_qty as sold_quantity 
    FROM {staging_silver_table}
""")

In [0]:
gold_delta = DeltaTable.forName(spark, gold_table) 
gold_delta.alias("source").merge(
    df_gold.alias("gold"), 
    "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code" 
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute() 

print(f" Incremental daily data merged into Child Gold table: {gold_table}")

In [0]:
df_incremental_dates = spark.sql(f"SELECT order_placement_date as date FROM {staging_silver_table}") 

incremental_month_df = df_incremental_dates.select(
    F.trunc("date", "MM").alias("start_month")
).distinct() 

incremental_month_df.createOrReplaceTempView("incremental_months")

In [0]:
monthly_table = spark.sql(f"""
    SELECT date, product_code, customer_code, sold_quantity
    FROM {gold_table} sbf
    INNER JOIN incremental_months m
        ON trunc(sbf.date, 'MM') = m.start_month
""") 

df_monthly_recalc = (
    monthly_table
    .withColumn("month_start", F.trunc("date", "MM")) 
    .groupBy("month_start", "product_code", "customer_code") 
    .agg(F.sum("sold_quantity").alias("sold_quantity")) 
    .withColumnRenamed("month_start", "date") 
)

In [0]:
parent_fact_table = f"{catalog}.{gold_schema}.fact_orders" 
gold_parent_delta = DeltaTable.forName(spark, parent_fact_table) 

gold_parent_delta.alias("parent_gold").merge(
    df_monthly_recalc.alias("child_gold"), 
    "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code" 
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print(f" Parent Fact Table perfectly updated with recalculated monthly totals!")

spark.sql(f"DROP TABLE IF EXISTS {catalog}.{bronze_schema}.staging_{data_source}") 
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{silver_schema}.staging_{data_source}") 
print(" Staging tables cleaned up.")